In [17]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

import os
from enum import Enum

In [18]:
load_dotenv()

True

In [19]:
LLM_API_URL = os.environ["LLM_API_URL"]
LLM_API_TOKEN = os.environ["LLM_API_TOKEN"]
MODEL = os.environ["MODEL"]

In [20]:
# LLM_API_URL = os.environ["LMSTUDIO_BASE_URL"]
# LLM_API_TOKEN = os.environ["LM_API_TOKEN"]
# MODEL = "gemma-4-26B"

In [21]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)

# Modélisation du monde

In [22]:
VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [23]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [3, 1, 0, 0, 2, 0, 3], # (1, 1) # (1, 4) # (1, 6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3], # (5, 6)
    [0, 0, 0, 0, 0, 0, 0],
])
initial_map

array([[0, 0, 0, 0, 0, 0, 0],
       [3, 1, 0, 0, 2, 0, 3],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0]])

# Règles de game design (décisions figées)

- **Nombre de pièces à ramasser : UNE seule** pour le moment (la partie s'arrête dès la première pièce ramassée).
  → Justification : on garde le scope minimal pour avancer sur la partie data engineering (benchmark, pipeline dbt-duckdb) en priorité. Si le temps le permet, on reviendra étendre à "toutes les pièces" pour enrichir le benchmark.
- **Combats : non autorisés.** Les ennemis sont des obstacles géométriques (case infranchissable via `allowed_move`), pas des adversaires à combattre.
  → Justification : un système de combat (PV, aléatoire de victoire...) ajoute de la complexité sans rapport avec l'objectif du projet (mesurer la qualité de navigation/décision du LLM).
- **Layout de départ : celui défini dans `initial_map`** (un ennemi entre le joueur et une pièce proche, une pièce "leurre" plus loin, une autre isolée).
  → Justification : il oblige déjà le joueur à contourner un obstacle pour atteindre la pièce la plus proche, ce qui teste la vraie capacité de décision du LLM (pas juste "aller tout droit").


# Couche de contrat

In [24]:
H = "HAUT"
B = "BAS"
G = "GAUCHE"
D = "DROITE"

# H = "TOP"
# B = "DOWN"
# G = "LEFT"
# D = "RIGHT"

class Direction(str, Enum):
    HAUT       = H
    BAS        = B
    GAUCHE     = G
    DROITE     = D


class PlayerDecision(BaseModel):
    # directionJustification: str
    direction: Direction

MOVES = {
    H:   (-1,   0),
    B:   ( 1,   0),
    G:   ( 0,  -1),
    D:   ( 0,   1),
}

# Moteur de perception

In [25]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [26]:
def compute_distances(entities_positions, reference_pos):
    if (len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
 
    return np.round(distances, 2)

In [27]:
def perception(world_map):

    player_position = localize(world_map, PLAYER)[0]
    golds_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    golds_distances = compute_distances(golds_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    # perception directionnelle → delta signé vers l'or le plus proche
    nearest_gold_delta = {"row": 0, "col": 0}
    if len(golds_positions) > 0:
        nearest_idx = np.argmin(golds_distances)
        nearest_gold_pos = golds_positions[nearest_idx]
        delta = nearest_gold_pos - player_position
        nearest_gold_delta = {"row": int(delta[0]), "col": int(delta[1])}

    return {
        "ennemies_distances": ennemies_distances.tolist(),
        "ennemies_count": len(ennemies_distances),
        "golds_distances": golds_distances.tolist(),
        "golds_count": len(golds_distances),
        "nearest_gold_delta": nearest_gold_delta,
    }

In [28]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('-----------------------------------------------------')

# Moteur de déplacement

In [29]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] in (VOID, GOLD)

In [30]:
def move(world_map: np.ndarray, old_pos, new_pos):
    move_result = {
        "gold_collected": False,
        "new_pos": old_pos
    }

    if not allowed_move(world_map, new_pos):
        return move_result

    entity = world_map[old_pos[0], old_pos[1]]
    target = world_map[new_pos[0], new_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    move_result["new_pos"] = new_pos

    if target == GOLD:
        move_result["gold_collected"] = True

    return move_result

# Moteur de décision

In [31]:
def decide(player_perception) -> PlayerDecision | None:
    prompt = f"""
    # Contexte
    - Tu es un joueur qui veut ramasser de l'or

    # Objectif
    - Trouve le plus court chemin vers l'or le plus proche, en évitant les ennemis

    # Perception
    {player_perception}

    # Aide à la lecture de "nearest_gold_delta"
    - row négatif → l'or le plus proche est plus HAUT que toi
    - row positif → l'or le plus proche est plus BAS que toi
    - col négatif → l'or le plus proche est plus à GAUCHE que toi
    - col positif → l'or le plus proche est plus à DROITE que toi
    - Utilise "ennemies_distances" pour éviter de te rapprocher trop d'un ennemi
    - Utilise "move_history" pour éviter de répéter les mêmes allers-retours
    """

    # print(prompt)
    print(str(player_perception))

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PlayerDecision,
        temperature=0.2
    )

    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [32]:
import uuid
from datetime import datetime, timezone
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

BRONZE_DIR = Path("data/bronze/turns")
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# Identifie le jeu de règles actif au moment du run (cf. cellule "Règles de game design").
# Permet de regrouper/filtrer les runs de façon cohérente si les règles évoluent plus tard,
# sans avoir à réinterpréter un schéma qui aurait changé de sens silencieusement.
RULESET_VERSION = "single_gold_no_combat_v1"

In [33]:
def game_loop(world_map: np.ndarray, max_turns = 10):
    world_map = world_map.copy()
    move_history = []
    golds_collected = 0
    turn_logs = []

    run_id = str(uuid.uuid4())
    run_started_at = datetime.now(timezone.utc).isoformat()

    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")
        show_map(world_map)

        player_pos = localize(world_map, PLAYER)[0]

        p = perception(world_map)
        p["move_history"] = move_history[-5:]

        decision: PlayerDecision | None = decide(p)
        llm_direction = decision.direction.value if decision is not None else None

        new_pos = tuple(int(x) for x in player_pos)
        gold_collected = False

        if decision is not None:
            print(f"\t → LLM decision: {llm_direction}")

            move_history.append(llm_direction)

            d_row, d_col = MOVES[llm_direction]
            target_pos = (player_pos[0] + d_row, player_pos[1] + d_col)

            move_result = move(world_map, player_pos, target_pos)
            gold_collected = move_result["gold_collected"]
            new_pos = tuple(int(x) for x in move_result["new_pos"])

            if gold_collected:
                golds_collected += 1
                print("FOUND GOLD !!!")

        turn_logs.append({
            "run_id": run_id,
            "run_started_at": run_started_at,
            "model": MODEL,
            "ruleset_version": RULESET_VERSION,
            "max_turns": max_turns,
            "turn": turn + 1,
            "player_row": int(player_pos[0]),
            "player_col": int(player_pos[1]),
            "golds_count": p["golds_count"],
            "nearest_gold_distance": min(p["golds_distances"]) if p["golds_distances"] else None,
            "ennemies_count": p["ennemies_count"],
            "nearest_ennemy_distance": min(p["ennemies_distances"]) if p["ennemies_distances"] else None,
            "nearest_gold_delta_row": p["nearest_gold_delta"]["row"],
            "nearest_gold_delta_col": p["nearest_gold_delta"]["col"],
            "llm_direction": llm_direction,
            "new_row": new_pos[0],
            "new_col": new_pos[1],
            "gold_collected": gold_collected,
        })

        if gold_collected:
            break

    table = pa.Table.from_pylist(turn_logs)
    bronze_path = BRONZE_DIR / f"run_{run_id}.parquet"
    pq.write_table(table, bronze_path)
    print(f"\n[bronze] {len(turn_logs)} tours écrits → {bronze_path}")

    return {
        "run_id": run_id,
        "golds_collected": golds_collected,
        "turns_played": turn + 1,
        "move_history": move_history,
        "bronze_path": str(bronze_path),
    }

In [34]:
game_loop(world_map=initial_map, max_turns=10)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
💰	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
{'ennemies_distances': [3.0], 'ennemies_count': 1, 'golds_distances': [1.0, 5.0, 6.4], 'golds_count': 3, 'nearest_gold_delta': {'row': 0, 'col': -1}, 'move_history': []}
	 → LLM decision: DROITE

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
💰	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
{'ennemies_distances': [2.0], 'ennemies_count': 1, 'golds_distances': [2.0, 4.0, 5.66], 'golds_count': 3, 'nearest_gold_delta': {'row': 0, 'col': -2}, 'move_history': ['DROITE']}
	 → LLM decision: HAUT

 =================== [Turn 3] ===================
·	·	👤	·	·	·	·
💰	·	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
------------------------------------------------

{'run_id': '5431db2a-93f6-48b7-8644-ee06e703a73c',
 'golds_collected': 0,
 'turns_played': 10,
 'move_history': ['DROITE',
  'HAUT',
  'HAUT',
  'HAUT',
  'HAUT',
  'GAUCHE',
  'DROITE',
  'DROITE',
  'DROITE',
  'DROITE'],
 'bronze_path': 'data\\bronze\\turns\\run_5431db2a-93f6-48b7-8644-ee06e703a73c.parquet'}